In [1]:
import pickle

from neurotools import models
from neurotools import modules
import torch
import torchvision
from torchvision.datasets.mnist import FashionMNIST, MNIST
import networkx as nx
from matplotlib import pyplot as plt

In [2]:
batch_size_train = 1
batch_size_test = 1
train_loader = torch.utils.data.DataLoader(
    MNIST('./tmp/files/', train=True, download=True,
                               transform=torchvision.transforms.Compose([
                                   torchvision.transforms.ToTensor(),
                                   torchvision.transforms.Normalize(
                                       (0.1307,), (0.3081,))
                               ])),
    batch_size=batch_size_train, shuffle=True)

test_loader = torch.utils.data.DataLoader(
    MNIST('./tmp/files/', train=False, download=True,
                               transform=torchvision.transforms.Compose([
                                   torchvision.transforms.ToTensor(),
                                   torchvision.transforms.Normalize(
                                       (0.1307,), (0.3081,))
                               ])),
    batch_size=batch_size_test, shuffle=True)

In [3]:
revnet = models.ElegantReverbNetwork(num_nodes=3, input_nodes=(0,), node_shape=(1, 2, 28, 28), edge_module=modules.ElegantReverb, device='cuda', mask=torch.ones((4, 4), device='cuda'))
revnet_decoder = torch.nn.Sequential(torch.nn.MaxPool2d(2),
                                     torch.nn.Conv2d(kernel_size=14, in_channels=1, out_channels=10, device="cuda", bias=False))

In [4]:
present_frames = 5
optimizer = torch.optim.Adam(lr=.01, params=list(revnet_decoder.parameters()) + list(revnet.parameters()))
ce_loss = torch.nn.NLLLoss()

In [7]:
history = []
for epoch in range(2000):
    targets = []
    local_history = []
    optimizer.zero_grad()
    revnet.detach(reset_intrinsic=False)
    for i, (stim, target) in enumerate(train_loader):
        if i > 300:
            break
        for _ in range(present_frames):
            revnet(stim.to("cuda"))
        decode_input = revnet.states[2, 0, :, :][None, None, :, :].clone()
        y_hat = revnet_decoder(decode_input)
        y_hat = y_hat.view(1, 10)
        y_hat = torch.log_softmax(y_hat, dim=1)
        target = target.long().to("cuda")
        # insert loss information
        local_history.append(y_hat.clone())
        targets.append(target.clone())
    lh = torch.cat(local_history[-100:])
    lt = torch.cat(targets[-100:])
    loss = ce_loss(lh, lt)
    acc = torch.count_nonzero(torch.argmax(lh, dim=1) == lt)
    print("EPOCH:", epoch)
    print(loss.detach().cpu().item())
    print(acc.detach().cpu().item())
    loss.backward()
    optimizer.step()
    history.append(acc.detach().cpu().item())

plt.plot(history)
plt.show()

EPOCH: 0
3.8571617603302
14
EPOCH: 1
3.6314687728881836
17
EPOCH: 2
4.154367446899414
16
EPOCH: 3
3.743495464324951
11
EPOCH: 4
3.1098570823669434
11
EPOCH: 5
4.001035690307617
9
EPOCH: 6
4.766395092010498
11
EPOCH: 7
5.372369766235352
9
EPOCH: 8
4.765527725219727
9
EPOCH: 9
5.688640117645264
9
EPOCH: 10
6.641534328460693
7
EPOCH: 11
5.270136833190918
13
EPOCH: 12
4.227010726928711
12
EPOCH: 13
4.852004528045654
11
EPOCH: 14
5.648622512817383
10
EPOCH: 15
4.058018684387207
14
EPOCH: 16
5.914982318878174
10
EPOCH: 17
6.729134559631348
9
EPOCH: 18
6.788455963134766
5
EPOCH: 19
7.202186107635498
15
EPOCH: 20
6.736647129058838
11
EPOCH: 21
5.368112087249756
11
EPOCH: 22
5.475518226623535
7
EPOCH: 23
5.321463584899902
10
EPOCH: 24
4.887434005737305
12
EPOCH: 25
4.124632358551025
4
EPOCH: 26
5.953056812286377
7
EPOCH: 27
4.609354019165039
11
EPOCH: 28
5.0965375900268555
19
EPOCH: 29
5.517277717590332
6
EPOCH: 30
4.505213737487793
10
EPOCH: 31
4.572515487670898
14
EPOCH: 32
5.7192583084106445

KeyboardInterrupt: 

In [6]:
import pickle
with open("./models/l2l_out_mk3.pkl", 'wb') as f:
    pickle.dump((revnet, revnet_decoder), f)

In [6]:
import pickle
with open("./models/l2l_out_mk3.pkl", 'rb') as f:
    revnet, revnet_decoder = pickle.load(f)

In [7]:
# learn without optimization
revnet.detach(reset_intrinsic=True)
local_history = []
targets = []
with torch.no_grad():
    for i, (stim, target) in enumerate(test_loader):
        for _ in range(present_frames):
            revnet(stim.to("cuda"))
        decode_input = revnet.states[0, 0, :, :][None, None, :, :].clone()
        y_hat = revnet_decoder(decode_input)
        y_hat = y_hat.view(1, 10)
        y_hat = torch.log_softmax(y_hat, dim=1)
        target = target.long().to("cuda")
        local_history.append(y_hat.clone())
        targets.append(target.clone())

In [9]:
phats = torch.cat(local_history)
t = torch.cat(targets)
f50_acc = torch.count_nonzero(torch.argmax(phats[:10], dim=1) == t[:10]) / 10
l100_acc = torch.count_nonzero(torch.argmax(phats[-100:], dim=1) == t[-100:]) / 100
print("Initial Acc: ", f50_acc.detach().cpu().item())
print("Final Acc: ", l100_acc.detach().cpu().item())

Initial Acc:  0.6000000238418579
Final Acc:  0.5399999618530273


In [11]:
other_loader = torch.utils.data.DataLoader(
    MNIST('./tmp/files/', train=True, download=True,
                               transform=torchvision.transforms.Compose([
                                   torchvision.transforms.ToTensor(),
                                   torchvision.transforms.Normalize(
                                       (0.1307,), (0.3081,))
                               ])),
    batch_size=batch_size_train, shuffle=True)

  0%|          | 0/9912422 [00:00<?, ?it/s]

Extracting ./tmp/files/MNIST/raw/train-images-idx3-ubyte.gz to ./tmp/files/MNIST/raw



  0%|          | 0/28881 [00:00<?, ?it/s]

Extracting ./tmp/files/MNIST/raw/train-labels-idx1-ubyte.gz to ./tmp/files/MNIST/raw



  0%|          | 0/1648877 [00:00<?, ?it/s]

Extracting ./tmp/files/MNIST/raw/t10k-images-idx3-ubyte.gz to ./tmp/files/MNIST/raw



  0%|          | 0/4542 [00:00<?, ?it/s]

Extracting ./tmp/files/MNIST/raw/t10k-labels-idx1-ubyte.gz to ./tmp/files/MNIST/raw



In [15]:
# learn entirely different set without numerical optimization
revnet.detach(reset_intrinsic=True)
local_history = []
targets = []
with torch.no_grad():
    for i, (stim, target) in enumerate(other_loader):
        if i > 10000:
            break
        for _ in range(present_frames):
            revnet(stim.to("cuda"))
        decode_input = revnet.states[0, 0, :, :][None, None, :, :].clone()
        y_hat = revnet_decoder(decode_input)
        y_hat = y_hat.view(1, 10)
        y_hat = torch.log_softmax(y_hat, dim=1)
        target = target.long().to("cuda")
        local_history.append(y_hat.clone())
        targets.append(target.clone())

In [18]:
phats = torch.cat(local_history)
t = torch.cat(targets)
f50_acc = torch.count_nonzero(torch.argmax(phats[:50], dim=1) == t[:50]) / 50
l100_acc = torch.count_nonzero(torch.argmax(phats[-500:], dim=1) == t[-500:]) / 500
print("Initial Acc: ", f50_acc.detach().cpu().item())
print("Final Acc: ", l100_acc.detach().cpu().item())

Initial Acc:  0.07999999821186066
Final Acc:  0.07800000160932541
